# Visual Analytics

## Assignment 3

**Instructor:** Dr. Marco D'Ambros  
**TAs:** Carmen Armenti, Mattia Giannaccari

**Contacts:** marco.dambros@usi.ch, carmen.armenti@usi.ch, mattia.giannaccari@usi.ch

**Due Date:** May 16, 2025 @ 23:55

---
The goal of this assignment is to use **Spark (PySpark)** and **Polars** in Jupyter notebooks.  
The files `trip_data.csv`, `trip_fare.csv`, and `nyc_boroughs.geojson` are available in the provided folder: [Assignment3-data](https://usi365-my.sharepoint.com/:f:/g/personal/armenc_usi_ch/Ejp7sb8QAMROoWe0XUDcAkMBoqUFk-w2Vgroup025NhAww?e=2I7SMC).

You may clean the data as needed; however, please note that specific data cleaning steps will be required in **Exercise 5**. If you choose to clean the data before Exercise 5, make sure to retain the **original dataset** for use with the Polars exercises.

- Use **Spark** to solve **Exercises 1–4**
- Use **Polars** to solve **Exercises 5–8**

You are encouraged to use [Spark window functions](https://spark.apache.org/docs/latest/sql-ref-syntax-qry-select-window.html) whenever appropriate.

Please name your notebook file as `SurnameName_Assignment3.ipynb`

## Spark

### Exercise 1
Join the `trip_data` and `trip_fare` dataframes into one and consider only data on 2013-01-01. Please specify the number of rows obtained after joining the 2 datasets.

In [43]:
from datetime import timedelta

from pyspark.sql import SparkSession
from pyspark.sql.functions import to_date

In [44]:
# Initialize Spark Session
spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark.sparkContext.setLogLevel('ERROR')

In [45]:
# Specify the paths to your CSV files
trip_data_path = "data/trip_data.csv"
trip_fare_path = "data/trip_fare.csv"

# Read the CSV files into DataFrames
trip_data = spark.read.csv(trip_data_path, header=True, inferSchema=True)
trip_fare = spark.read.csv(trip_fare_path, header=True, inferSchema=True)

# Function to strip whitespace from column names
def strip_column_names(df):
    return df.select(*[df[col].alias(col.strip()) for col in df.columns])

# Strip whitespace from column names of both DataFrames
trip_data = strip_column_names(trip_data)
trip_fare = strip_column_names(trip_fare)

# Print the schemas of the DataFrames to verify they are read correctly
print("Schema of trip_data:")
trip_data.printSchema()
#print(f"Number of rows: {trip_data.count()}")
print("\nSchema of trip_fare:")
trip_fare.printSchema()
#print(f"Number of rows: {trip_fare.count()}")

# Filter both dataframes for the date '2013-01-01'
# We can directly compare the date part of the timestamp
trip_data_jan01 = trip_data.filter((to_date(trip_data["pickup_datetime"]) == "2013-01-01") & (to_date(trip_data["dropoff_datetime"]) == "2013-01-01"))
trip_fare_jan01 = trip_fare.filter(to_date(trip_fare["pickup_datetime"]) == "2013-01-01")

#print(trip_data_jan01.count())
#subset_columns = ['medallion', 'hack_license', 'vendor_id']
#print(trip_fare_jan01.dropDuplicates(subset=subset_columns).count())

# Perform the join operation
# Using the common columns 'medallion' and the pickup datetime columns (with the leading space for trip_fare)
trip_filtered_df = trip_data_jan01.join(trip_fare_jan01, ["medallion", 'hack_license', 'vendor_id' ,"pickup_datetime"], "inner")

# Get the number of rows in the joined dataframe
row_count = trip_filtered_df.count()

# Print the number of rows
print(f"\nThe number of rows in the joined dataframe for 2013-01-01 is: {row_count}")

Schema of trip_data:
root
 |-- medallion: string (nullable = true)
 |-- hack_license: string (nullable = true)
 |-- vendor_id: string (nullable = true)
 |-- rate_code: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_time_in_secs: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)


Schema of trip_fare:
root
 |-- medallion: string (nullable = true)
 |-- hack_license: string (nullable = true)
 |-- vendor_id: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- surcharge: double (nulla


The number of rows in the joined dataframe for 2013-01-01 is: 410816


In [46]:
trip_filtered_df.schema

StructType([StructField('medallion', StringType(), True), StructField('hack_license', StringType(), True), StructField('vendor_id', StringType(), True), StructField('pickup_datetime', TimestampType(), True), StructField('rate_code', IntegerType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('dropoff_datetime', TimestampType(), True), StructField('passenger_count', IntegerType(), True), StructField('trip_time_in_secs', IntegerType(), True), StructField('trip_distance', DoubleType(), True), StructField('pickup_longitude', DoubleType(), True), StructField('pickup_latitude', DoubleType(), True), StructField('dropoff_longitude', DoubleType(), True), StructField('dropoff_latitude', DoubleType(), True), StructField('payment_type', StringType(), True), StructField('fare_amount', DoubleType(), True), StructField('surcharge', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', Do

### Exercise 2
Provide a graphical representation to compare the average fare amount for trips _within_ and _across_ all the boroughs. You may want to have a look at: https://docs.bokeh.org/en/latest/docs/user_guide/topics/categorical.html#categorical-heatmaps

In [47]:
import geopandas as gpd
from math import pi

from bokeh.plotting import figure, show, reset_output, output_notebook

reset_output()
output_notebook()

Loading BokehJS ...

In [48]:
borough_gdf = gpd.read_file('./data/nyc-boroughs.geojson')

In [49]:
trip_filtered_pd_df = trip_filtered_df.toPandas()

pickup_gdf = gpd.GeoDataFrame(
    trip_filtered_pd_df,
    geometry=gpd.points_from_xy(trip_filtered_pd_df['pickup_longitude'], trip_filtered_pd_df['pickup_latitude']),
    crs=borough_gdf.crs
)

dropoff_gdf = gpd.GeoDataFrame(
    trip_filtered_pd_df,
    geometry=gpd.points_from_xy(trip_filtered_pd_df['dropoff_longitude'], trip_filtered_pd_df['dropoff_latitude']),
    crs=borough_gdf.crs
)

pickup_with_borough = gpd.sjoin(pickup_gdf, borough_gdf[['borough', 'geometry']], how='left', predicate='within')
dropoff_with_borough = gpd.sjoin(dropoff_gdf, borough_gdf[['borough', 'geometry']], how='left', predicate='within')

trip_filtered_pd_df['pickup_borough'] = pickup_with_borough['borough']
trip_filtered_pd_df['dropoff_borough'] = dropoff_with_borough['borough']

trip_filtered_df = spark.createDataFrame(trip_filtered_pd_df)

In [50]:
from bokeh.models import BasicTicker, PrintfTickFormatter, ColumnDataSource
from bokeh.plotting import figure, show
from bokeh.transform import linear_cmap
from pyspark.sql.functions import min as spark_min, max as spark_max
import pandas as pd  # Import pandas

# Assuming trip_filtered_df and borough_gdf are Spark DataFrames
trip_group_df_spark = trip_filtered_df.groupBy(['pickup_borough', 'dropoff_borough']).avg('fare_amount').withColumnRenamed('avg(fare_amount)', 'avg_fare')

# Convert the Spark DataFrame to a Pandas DataFrame
trip_group_df = trip_group_df_spark.toPandas()

# Get unique boroughs as a Python list
pickup_borough = borough_gdf['borough'].unique().tolist()
dropoff_borough = borough_gdf['borough'].unique().tolist()
dropoff_borough = dropoff_borough[::-1] # Reverse the list to have the same order

# Calculate min and max of average fare
min_fare = trip_group_df['avg_fare'].min()
max_fare = trip_group_df['avg_fare'].max()

# this is the colormap from the original NYTimes plot
colors = ["#75968f", "#a5bab7", "#c9d9d3", "#e2e2e2", "#dfccce", "#ddb7b1", "#cc7878", "#933b41", "#550b1d"]

TOOLS = "hover,save,pan,box_zoom,reset,wheel_zoom"
TOOLTIPS = [
    ('Pickup Borough', '@pickup_borough'),
    ('Dropoff Borough', '@dropoff_borough'),
    ('Average Fare Amount', '@avg_fare{0.2f}')
]

p = figure(title="Average Fare Amount for Pickup and Dropoff Boroughs",
           x_range=pickup_borough, y_range=dropoff_borough,
           x_axis_location="above", width=900, height=400,
           tools=TOOLS, toolbar_location='below',
           tooltips=TOOLTIPS)

p.grid.grid_line_color = None
p.axis.axis_line_color = None
p.axis.major_tick_line_color = None
p.axis.major_label_text_font_size = "7px"
p.axis.major_label_standoff = 0
p.xaxis.major_label_orientation = pi / 3

r = p.rect(x="pickup_borough", y="dropoff_borough", width=1, height=1, source=ColumnDataSource(trip_group_df),
           fill_color=linear_cmap("avg_fare", colors, low=min_fare, high=max_fare),
           line_color=None)

p.add_layout(r.construct_color_bar(
    major_label_text_font_size="7px",
    ticker=BasicTicker(desired_num_ticks=len(colors)),
    formatter=PrintfTickFormatter(format="%.2f"), # Changed formatter
    label_standoff=6,
    border_line_color=None,
    padding=5,
), 'right')

show(p)

### Exercise 3
Consider only Manhattan, Bronx and Brooklyn boroughs. Then create a dataframe that shows the total number of trips *within* the same borough and *across* all the other boroughs mentioned before (Manhattan, Bronx, and Brooklyn) where the passengers are more or equal than 3.

For example, for Manhattan borough you should consider the total number of the following trips:
- Manhattan → Manhattan
- Manhattan → Bronx
- Manhattan → Brooklyn

You should then do the same for Bronx and Brooklyn boroughs.

In [51]:
from pyspark.sql.functions import col, when, count

# Define the boroughs of interest
boroughs_of_interest = ["Manhattan", "Bronx", "Brooklyn"]

# Filter the joined_df for trips starting and ending in the boroughs of interest and with at least 3 passengers
filtered_trips = trip_filtered_df.filter(
    (col("pickup_borough").isin(boroughs_of_interest)) &
    (col("dropoff_borough").isin(boroughs_of_interest)) &
    (col("passenger_count") >= 3)
).groupBy(['pickup_borough', 'dropoff_borough']).count()

filtered_trips

pickup_borough,dropoff_borough,count
Brooklyn,Manhattan,1326
Brooklyn,Bronx,10
Manhattan,Manhattan,62391
Brooklyn,Brooklyn,2014
Manhattan,Brooklyn,2762
Manhattan,Bronx,527
Bronx,Manhattan,55
Bronx,Bronx,103
Bronx,Brooklyn,2


### Exercise 4
Create a dataframe where each row represents a driver, and there is one column per borough.
For each driver-borough, the dataframe provides the maximum number of consecutive trips
for the given driver, within the given borough. Please consider only trips which were payed by card. 

For example, if for driver A we have (sorted by time):
- Trip 1: Bronx → Bronx
- Trip 2: Bronx → Bronx
- Trip 3: Bronx → Manhattan
- Trip 4: Manhattan → Bronx.
    
The maximum number of consecutive trips for Bronx is 2.

In [52]:
pd.options.mode.chained_assignment = None
trip_filtered_df = trip_filtered_df.sort(['hack_license', 'pickup_datetime'], ascending=True)

trip_drivers_df = trip_filtered_pd_df[['hack_license', 'pickup_borough', 'dropoff_borough', 'payment_type']]
boroughs = borough_gdf['borough'].unique()
payment = 'CRD'

for borough in boroughs:
    pickup_filter = trip_drivers_df['pickup_borough'] == borough
    dropoff_filter = trip_drivers_df['dropoff_borough'] == borough
    payment_filter = trip_drivers_df['payment_type'] == payment

    condition = (pickup_filter & dropoff_filter & payment_filter).to_numpy()
    condition = condition.astype(int)
    condition = condition.tolist()
    borough_count = 1 if condition[0] else 0

    for i in range(1, len(condition)):
        if condition[i]:
            borough_count += 1
        else:
            borough_count = 0

        condition[i] = borough_count

    trip_drivers_df[borough] = condition

trip_drivers_df = trip_drivers_df.groupby(['hack_license']).agg(
    {'Manhattan': 'max', 'Brooklyn': 'max', 'Bronx': 'max', 'Queens': 'max', 'Staten Island': 'max'}
).reset_index()

trip_drivers_df

,hack_license,Manhattan,Brooklyn,Bronx,Queens,Staten Island
0,0002555BBE359440D6CEB34B699D3932,1,0,0,1,0
1,000A4EBF1CEB9C6BD9978D4362493C6E,0,1,0,0,0
2,000B8D660A329BBDBF888500E4BD8B98,2,1,0,0,0
3,000CCA239BFDC0ABE2895AC9086C4290,3,0,0,0,0
4,00117D7CCD47D125E77163A7AC2C66EB,2,0,0,0,0
...,...,...,...,...,...,...
19826,FFF20BA1518E14B3B23F79DDDE1CA7E6,2,0,0,0,0
19827,FFF5AD65C673251C1F275CF5B43EC414,3,1,0,0,0
19828,FFF657CFEC6A06384C97ACB500916913,1,0,0,0,0
19829,FFF909B1353148850AD3E40BB878618B,2,0,0,0,0


## Polars

### Exercise 5

Please work on the merged dataset of trips and fares and perform the following data cleaning tasks:

1. Remove trips with invalid locations (i.e. not in New York City);
3. Remove trips with invalid amounts:
    - Total amount must be greater than zero;
    - Total amount must correspond to the sum of all the other amounts.
5. Remove trips with invalid time:
    - Pick-up before drop-off;
    - Valid duration.

After each data cleaning task, report how many rows where removed. Finally report:
- Are there **duplicate trips**?
- How many trips remain after cleaning?

In [53]:
from typing import Any
from pathlib import Path
import polars as pl
import re
import json

In [54]:
def parse_spark_type(spark_type):
    """Convert Spark types to Polars types."""
    if spark_type.startswith("array<"):
        inner = spark_type[6:-1]
        return pl.List(parse_spark_type(inner))
    elif spark_type.startswith("struct<"):
        inner = spark_type[7:-1]
        fields = []
        for part in inner.split(','):
            k, v = part.split(':')
            fields.append((k.strip(), parse_spark_type(v.strip())))
        return pl.Struct(fields)
    elif spark_type == "string":
        return pl.Utf8
    elif spark_type in ("int", "integer"):
        return pl.Int32
    elif spark_type == "long":
        return pl.Int64
    elif spark_type == "double":
        return pl.Float64
    elif spark_type == "float":
        return pl.Float32
    elif spark_type == "boolean":
        return pl.Boolean
    elif spark_type == "timestamp":
        return pl.Datetime
    elif spark_type == "date":
        return pl.Date
    else:
        return pl.Object

def parse_spark_schema(schema_str):
    """Parse printSchema string into Polars schema dict."""
    lines = schema_str.strip().splitlines()
    schema = {}
    for line in lines:
        match = re.search(r"\|-- (\w+): ([^ ]+)", line)
        if match:
            col_name, col_type = match.groups()
            schema[col_name] = parse_spark_type(col_type)
    return schema


tripdata_polar = pl.read_csv('./data/trip_data.csv', schema=parse_spark_schema(trip_data._jdf.schema().treeString()))
tripfare_polar = pl.read_csv('./data/trip_fare.csv', schema=parse_spark_schema(trip_fare._jdf.schema().treeString()))

tripdata_polar = tripdata_polar.rename({col: col.strip() for col in tripdata_polar.columns})
tripfare_polar = tripfare_polar.rename({col: col.strip() for col in tripfare_polar.columns})

trip_polar = tripdata_polar.join(
    tripfare_polar,
    on=['medallion', 'hack_license', 'vendor_id', 'pickup_datetime'],
    how='inner'
)

In [55]:
def get_min_max_coordinates(
    geojson: dict[str, Any],
) -> tuple[float, float, float, float]:
    """
    Get the min/max coordinates from a geojson file.
    """
    # get all the coordinates from the geojson file
    coordinates: list[tuple[float, float]] = []
    for feature in geojson["features"]:
        coordinates.extend(feature["geometry"]["coordinates"][0])

    min_lon = min([point[0] for point in coordinates])
    max_lon = max([point[0] for point in coordinates])
    min_lat = min([point[1] for point in coordinates])
    max_lat = max([point[1] for point in coordinates])

    return min_lon, max_lon, min_lat, max_lat

initial_rows = len(trip_polar)
print(f"Initial rows: {initial_rows}")

nyc_boroughs_geojson_path = Path("./data/nyc-boroughs.geojson")

# read the geojson file
with nyc_boroughs_geojson_path.open("r") as f:
    json_data = f.read()
    nyc_boroughs_geo_data: dict[str, Any] = json.loads(json_data)

# get the min/max coordinates
min_lon, max_lon, min_lat, max_lat = get_min_max_coordinates(nyc_boroughs_geo_data)

trip_polar_filtered = trip_polar.filter(
    pl.col("total_amount") > 0,
    (pl.col('total_amount') == (
        pl.col('fare_amount') +
        pl.col('surcharge') +
        pl.col('mta_tax') +
        pl.col('tip_amount') +
        pl.col('tolls_amount')
    ))
)
print(f"Rows after removing invalid amounts: {len(trip_polar_filtered)}")

trip_polar_filtered = trip_polar_filtered.filter(
    pl.col("pickup_longitude") > min_lon,
    pl.col("pickup_longitude") < max_lon,
    pl.col("pickup_latitude") > min_lat,
    pl.col("pickup_latitude") < max_lat,
    pl.col("dropoff_longitude") > min_lon,
    pl.col("dropoff_longitude") < max_lon,
    pl.col("dropoff_latitude") > min_lat,
    pl.col("dropoff_latitude") < max_lat
)
print(f"Rows after removing invalid positions: {len(trip_polar_filtered)}")

trip_polar_filtered = trip_polar_filtered.filter(
    pl.col("trip_distance") > 0,
    (pl.col('dropoff_datetime') - pl.col('pickup_datetime')).dt.total_seconds() == pl.col('trip_time_in_secs'),
    (pl.col('dropoff_datetime') - pl.col('pickup_datetime')) < pl.duration(hours=12),
    pl.col('pickup_datetime') < pl.col('dropoff_datetime'),
    pl.col('passenger_count') != 0,
)
print(f"Rows after removing invalid trip times: {len(trip_polar_filtered)}")

# Check if there are duplicates
duplicates_df = trip_polar_filtered.group_by('hack_license', 'pickup_datetime', 'dropoff_datetime').agg(pl.len().alias('duplicated_trip'))

duplicates = duplicates_df.filter(pl.col('duplicated_trip') > 1).select(pl.col('duplicated_trip').sum().alias('total_duplicates')).to_dict()['total_duplicates'][0]
print(f"Rows of duplicated trips: {duplicates}")

end_rows = len(trip_polar_filtered)
print(f"In the end there are {end_rows}, rows removed are {initial_rows - end_rows}")
trip_polar_filtered

Initial rows: 14776615
Rows after removing invalid amounts: 14504027
Rows after removing invalid positions: 14211370
Rows after removing invalid trip times: 10341892
Rows of duplicated trips: 8
In the end there are 10341892, rows removed are 4434723


medallion,hack_license,vendor_id,rate_code,store_and_fwd_flag,pickup_datetime,dropoff_datetime,passenger_count,trip_time_in_secs,trip_distance,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,payment_type,fare_amount,surcharge,mta_tax,tip_amount,tolls_amount,total_amount
str,str,str,i32,str,datetime[μs],datetime[μs],i32,i32,f64,f64,f64,f64,f64,str,f64,f64,f64,f64,f64,f64
"""89D227B655E5C82AECF13C3F540D4C…","""BA96DE419E711691B9445D6A6307C1…","""CMT""",1,"""N""",2013-01-01 15:11:48,2013-01-01 15:18:10,4,382,1.0,-73.978165,40.757977,-73.989838,40.751171,"""CSH""",6.5,0.0,0.5,0.0,0.0,7.0
"""0BD7C8F5BA12B88E0B67BED28BEA73…","""9FD8F69F0804BDB5549F40E9DA1BE4…","""CMT""",1,"""N""",2013-01-06 00:18:35,2013-01-06 00:22:54,1,259,1.5,-74.006683,40.731781,-73.994499,40.75066,"""CSH""",6.0,0.5,0.5,0.0,0.0,7.0
"""0BD7C8F5BA12B88E0B67BED28BEA73…","""9FD8F69F0804BDB5549F40E9DA1BE4…","""CMT""",1,"""N""",2013-01-05 18:49:41,2013-01-05 18:54:23,1,282,1.1,-74.004707,40.73777,-74.009834,40.726002,"""CSH""",5.5,1.0,0.5,0.0,0.0,7.0
"""0B57B9633A2FECD3D3B1944AFC7471…","""CCD4367B417ED6634D986F573A552A…","""CMT""",1,"""N""",2013-01-07 12:39:18,2013-01-07 13:10:56,3,1898,10.7,-73.989937,40.756775,-73.86525,40.77063,"""CSH""",34.0,0.0,0.5,0.0,4.8,39.3
"""3349F919AA8AE5DC9C50A3773EA45B…","""7CE849FEF67514F080AF80D990F7EF…","""CMT""",1,"""N""",2013-01-10 15:42:29,2013-01-10 16:04:02,1,1293,3.2,-73.994911,40.723221,-73.971558,40.761612,"""CSH""",15.5,0.0,0.5,0.0,0.0,16.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""F33EF464441839C6F0DABAABBC93B4…","""313F66DD09C308EADA3B307F6B8CF7…","""CMT""",1,"""N""",2013-01-10 10:56:47,2013-01-10 11:05:52,1,545,1.4,-73.97541,40.759106,-73.96183,40.776527,"""CSH""",7.5,0.0,0.5,0.0,0.0,8.0
"""32201027CDC62D654DC3AD9747A07C…","""B8DDB9F8143017E22104050B26C2A6…","""CMT""",1,"""N""",2013-01-05 08:58:18,2013-01-05 09:05:56,1,458,3.2,-73.998901,40.734509,-73.96682,40.770138,"""CSH""",10.5,0.0,0.5,0.0,0.0,11.0
"""B33E71CD9E8FE1BE3B70FEB6E807DD…","""BAF57796E45D921BB23217E17A372F…","""CMT""",1,"""N""",2013-01-06 04:58:23,2013-01-06 05:11:24,1,781,3.3,-73.989029,40.759327,-73.953743,40.770672,"""CSH""",13.0,0.5,0.5,0.0,0.0,14.0


### Exercise 6

Compute the **total revenue** (total_amount) grouped by:
- Pick-up hour of the day (0–23)
- Passenger count (group >=6 into “6+”)

Create a heatmap where:
- X-axis = hour
- Y-axis = passenger count group
- Cell value = average revenue per trip

In [56]:
trip_polar_group = trip_polar_filtered \
    .with_columns([
        pl.when(pl.col('passenger_count') < 6)
          .then(pl.col('passenger_count').cast(pl.Utf8))  # convert to string for consistency
          .otherwise(pl.lit("6+"))
          .alias('passenger_count'),
        pl.col('pickup_datetime').dt.hour().cast(pl.Utf8).alias('pickup_hour')
    ]) \
    .group_by(['pickup_hour', 'passenger_count']) \
    .agg(
        pl.col('total_amount').mean().round(2).alias('avg_amount')
    )

trip_polar_group

pickup_hour,passenger_count,avg_amount
str,str,f64
"""10""","""1""",12.58
"""9""","""3""",13.25
"""0""","""2""",14.53
"""7""","""4""",14.05
"""5""","""3""",19.41
…,…,…
"""9""","""6+""",12.88
"""14""","""6+""",13.78
"""14""","""3""",13.66


In [57]:
# Ensure 'pickup_hour' is treated as categorical and sorted
x_range = sorted([str(h) for h in trip_polar_group['pickup_hour'].unique().to_list()], key=int)

# Ensure 'passenger_count' is sorted correctly
def sort_passenger_groups(groups):
    sorted_groups = []
    numeric_groups = []
    plus_group = None
    for group in groups:
        if isinstance(group, str) and group.endswith('+'):
            plus_group = group
        else:
            try:
                numeric_groups.append(int(group))
            except ValueError:
                pass
    sorted_groups.extend(sorted(numeric_groups))
    if plus_group:
        sorted_groups.append(plus_group)
    return [str(g) for g in sorted_groups]

y_range = sort_passenger_groups(trip_polar_group['passenger_count'].unique().to_list())

# this is the colormap from the original NYTimes plot
colors = ["#75968f", "#a5bab7", "#c9d9d3", "#e2e2e2", "#dfccce", "#ddb7b1", "#cc7878", "#933b41", "#550b1d"]

TOOLS = "hover,save,pan,box_zoom,reset,wheel_zoom"
TOOLTIPS = [
    ('Pickup Hour', '@pickup_hour'),
    ('Passenger Count', '@passenger_count'),
    ('Average Amount', '@avg_amount{0.2f}')
]

# Convert Polars DataFrame to Pandas for Bokeh
trip_group_df = trip_polar_group.to_pandas()

p = figure(title="Average Amount by Pickup Hour and Passenger Count",
           x_range=x_range, y_range=y_range, width=900, height=400,
           tools=TOOLS, toolbar_location='below',
           tooltips=TOOLTIPS)

p.grid.grid_line_color = None
p.axis.axis_line_color = None
p.axis.major_tick_line_color = None
p.axis.major_label_text_font_size = "7px"
p.axis.major_label_standoff = 0
p.xaxis.major_label_orientation = pi / 3

r = p.rect(x="pickup_hour", y="passenger_count", width=1, height=1, source=ColumnDataSource(trip_group_df),
           fill_color=linear_cmap("avg_amount", colors, low=trip_group_df['avg_amount'].min(), high=trip_group_df['avg_amount'].max()),
           line_color=None)

p.add_layout(r.construct_color_bar(
    major_label_text_font_size="7px",
    ticker=BasicTicker(desired_num_ticks=len(colors)),
    formatter=PrintfTickFormatter(format="%.2f"),
    label_standoff=6,
    border_line_color=None,
    padding=5,
), 'right')

show(p)

### Exercise 7

Define an "anomalous trip" as one that satisfies at least two of the following:
- Fare per mile is above the 95th percentile
- Tip amount > 100% of fare
- trip_time_in_secs is less than 60 seconds but distance is more than 1 mile

Create a dataframe of anomalous trips and:
- Report how many such trips exist
- Create a scatterplot to visualize the anomaly metrics
- Describe the visualization identifying groups and outliers

In [58]:
# Calculate fare per mile
df = trip_polar_filtered.with_columns(
    (pl.col('fare_amount') / pl.col('trip_distance')).alias('fare_per_mile')
)

# Calculate the 95th percentile of fare per mile
fare_per_mile_95th_percentile = df['fare_per_mile'].quantile(0.95)

# Define the conditions for an anomalous trip
condition_fare_per_mile = pl.col('fare_per_mile') > fare_per_mile_95th_percentile
condition_tip_over_fare = pl.col('tip_amount') > pl.col('fare_amount')
condition_short_time_long_distance = (pl.col('trip_time_in_secs') < 60) & (pl.col('trip_distance') > 1)

# Calculate anomaly score using Polars expressions
df = df.with_columns(
    (
        condition_fare_per_mile.cast(pl.Int8) +
        condition_tip_over_fare.cast(pl.Int8) +
        condition_short_time_long_distance.cast(pl.Int8)
    ).alias('anomaly_score')
)

# Filter for anomalous trips (meeting at least two conditions)
anomalous_trips_df = df.filter(pl.col('anomaly_score') >= 2)

# Report the number of anomalous trips
num_anomalous_trips = anomalous_trips_df.height
print(f"Number of anomalous trips: {num_anomalous_trips}")

# Create a simplified DataFrame for plotting
plot_df = anomalous_trips_df.select(['fare_per_mile', 'tip_amount', 'trip_time_in_secs', 'trip_distance'])

# Print the anomalous trips dataframe
print(anomalous_trips_df)

Number of anomalous trips: 1180
shape: (1_180, 23)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ medallion ┆ hack_lice ┆ vendor_id ┆ rate_code ┆ … ┆ tolls_amo ┆ total_amo ┆ fare_per_ ┆ anomaly_ │
│ ---       ┆ nse       ┆ ---       ┆ ---       ┆   ┆ unt       ┆ unt       ┆ mile      ┆ score    │
│ str       ┆ ---       ┆ str       ┆ i32       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│           ┆ str       ┆           ┆           ┆   ┆ f64       ┆ f64       ┆ f64       ┆ i8       │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ B6EA16667 ┆ 3C7E9CAAA ┆ VTS       ┆ 1         ┆ … ┆ 0.0       ┆ 65.0      ┆ 250.0     ┆ 2        │
│ 573BC4128 ┆ 0FEADD70A ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ A52741FEC ┆ E82B58180 ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ 5E7…      ┆ 823…      ┆           ┆   

In [65]:
from bokeh.models import HoverTool

# Create a simplified DataFrame for plotting
plot_df = anomalous_trips_df.select(['fare_per_mile', 'tip_amount', 'trip_time_in_secs', 'trip_distance', 'fare_amount']).to_pandas()

# Create the Bokeh plot
source = ColumnDataSource(plot_df)

p = figure(title="Anomalous Trip Metrics", x_axis_label="Fare per Mile", y_axis_label="Tip Amount",
           tools="pan,box_zoom,wheel_zoom,save,reset", width=800, height=600)

# Use scatter instead of circle and pass size as an argument
p.scatter(x="fare_per_mile", y="tip_amount", size=8, fill_color="green", fill_alpha=0.6, line_color="black", source=source)

# Add hover tooltips
hover = HoverTool(tooltips=[
    ("Fare per Mile", "@fare_per_mile{0.2f}"),
    ("Tip Amount", "@tip_amount{0.2f}"),
    ("Trip Time (secs)", "@trip_time_in_secs"),
    ("Trip Distance", "@trip_distance{0.2f}"),
    ("Fare Amount", "@fare_amount{0.2f}")
])
p.add_tools(hover)

show(p)

In the scatterplot we can easily identify some outliers, for example the point at bottom right that have the highest fare per mile and a very low tip amount. Also some other outliers show a very high tip amount for a low fare per mile. A big group is identified near the origin of the axis having a low tip amount and a low fare per mile

### Exercise 8
For each driver (hack_license), calculate the **total profit per hour worked**, where:
> profit = 0.7 * (fare_amount + tip_amount) when the trip starts between 7:01 AM and 7:00 PM\
> profit = 0.8 * (fare_amount + tip_amount) when the trip starts between 7:01PM and 7:00 AM

Estimate "hours worked" by summing trip_time_in_secs.

Plot a line chart showing the distribution of average profit per hour **for the top 10% drivers** in terms of total trips.

Which time of day offers **best earning efficiency**?

In [60]:
import polars as pl

trip_polar_filtered = trip_polar_filtered.with_columns(
    pl.col("pickup_datetime").cast(pl.Datetime).alias("pickup_datetime")
)

profit_calculation = pl.when((pl.col("pickup_datetime").dt.hour() >= 7) & (pl.col("pickup_datetime").dt.hour() <= 19)) \
    .then(0.7 * (pl.col("fare_amount") + pl.col("tip_amount"))) \
    .otherwise(0.8 * (pl.col("fare_amount") + pl.col("tip_amount")))

trip_polar_with_profit = trip_polar_filtered.with_columns(
    profit_calculation.alias("profit")
)

driver_profit = (
    trip_polar_with_profit.group_by("hack_license")
    .agg(
        pl.sum("profit").alias("total_profit"),
        (pl.sum("trip_time_in_secs") / 3600).alias("total_hours_worked"),
        (pl.sum("profit") / (pl.sum("trip_time_in_secs") / 3600)).alias(
            "profit_per_hour"
        ),
    )
)

print(driver_profit)

shape: (31_914, 4)
┌─────────────────────────────────┬──────────────┬────────────────────┬─────────────────┐
│ hack_license                    ┆ total_profit ┆ total_hours_worked ┆ profit_per_hour │
│ ---                             ┆ ---          ┆ ---                ┆ ---             │
│ str                             ┆ f64          ┆ f64                ┆ f64             │
╞═════════════════════════════════╪══════════════╪════════════════════╪═════════════════╡
│ 4C07BF65DD546E1847D449297CDC61… ┆ 1505.785     ┆ 33.168333          ┆ 45.398271       │
│ 6C244F5AA559CC18B2E60F1330EF7C… ┆ 7770.586     ┆ 168.9              ┆ 46.007022       │
│ 1B177C453E947B8373624F9A50D587… ┆ 2768.319     ┆ 49.866667          ┆ 55.514418       │
│ A97F39D0CAE8872EDB9DA4E7061D61… ┆ 2070.833     ┆ 40.070556          ┆ 51.679668       │
│ D9FFE788E58D7DC9296C9010984390… ┆ 2330.508     ┆ 49.265278          ┆ 47.305285       │
│ …                               ┆ …            ┆ …                  ┆ …        

In [61]:
import polars as pl
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
from bokeh.models import ColumnDataSource, HoverTool, Range1d

# 1. Compute trip counts per driver
trip_counts = (
    trip_polar_with_profit.group_by("hack_license")
    .agg(pl.len().alias("trip_count"))
)

# 2. Merge with driver_profit
driver_stats = driver_profit.join(trip_counts, on="hack_license")

# 3. Get top 10% of drivers by trip count
top_10_cutoff = int(0.1 * driver_stats.shape[0])
top_driver_ids = (
    driver_stats.sort("trip_count", descending=True)
    .head(top_10_cutoff)["hack_license"]
)

# 4. Filter trips to only those from top 10% drivers
top_driver_trips = trip_polar_with_profit.filter(
    pl.col("hack_license").is_in(top_driver_ids)
)

# 5. Extract hour of day
top_driver_trips = top_driver_trips.with_columns(
    pl.col("pickup_datetime").dt.hour().alias("pickup_hour")
)

# 6. Calculate total profit and time per hour
hourly_stats = (
    top_driver_trips.group_by("pickup_hour")
    .agg([
        pl.sum("profit").alias("total_profit"),
        pl.sum("trip_time_in_secs").alias("total_seconds"),
    ])
    .with_columns(
        (pl.col("total_profit") / (pl.col("total_seconds") / 3600)).alias("avg_profit_per_hour")
    )
    .sort("pickup_hour")
)

# 7. Prepare data for Bokeh
source = ColumnDataSource(data={
    "hour": hourly_stats["pickup_hour"].to_list(),
    "avg_profit_per_hour": hourly_stats["avg_profit_per_hour"].to_list()
})

# 8. Plot
output_notebook()

p = figure(title="Avg Profit per Hour of Day (Top 10% Drivers by Trips)",
           x_axis_label="Hour of Day",
           y_axis_label="Avg Profit per Hour ($)",
           x_range=(0, 23),
           tools="save",
           width=800, height=400)

p.y_range.start = 0
p.line(x="hour", y="avg_profit_per_hour", source=source, line_width=2, color="green")

# Add hover tool
hover = HoverTool(
    tooltips=[
        ("Hour", "@hour"),
        ("Avg Profit/Hr", "@avg_profit_per_hour{$0.00}")
    ],
    mode="vline"
)
p.add_tools(hover)

show(p)


Loading BokehJS ...

The best earning efficency is given at 5 am as is the highest value into the graph, showing the higher average of profit per hour